In [ ]:
# CELL 1 - Install dependencies
!pip install -q diffusers==0.38.0 transformers accelerate sentencepiece
!pip install -q fastapi uvicorn pyngrok "imageio[ffmpeg]"
!pip install -q opencv-python pillow numpy "huggingface_hub>=0.23.0"
import diffusers
print("Diffusers version:", diffusers.__version__)
print("LTXPipeline available:", hasattr(diffusers, "LTXPipeline"))
print("All dependencies installed!")

In [ ]:
# CELL 2 - Setup Kaggle Credentials
import os
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    kaggle_user = user_secrets.get_secret("KAGGLE_USERNAME")
except:
    kaggle_user = os.environ.get("KAGGLE_USERNAME", "__KAGGLE_USERNAME_PLACEHOLDER__")


In [ ]:
# CELL 3 - Load LTX model and start FastAPI server
import sys, os
PKGS = "/kaggle/working/pkgs"
if PKGS not in sys.path:
    sys.path.insert(0, PKGS)
import threading, uvicorn, torch, time, base64, traceback, requests
from fastapi import FastAPI
from fastapi.responses import JSONResponse
from pydantic import BaseModel
# Import LTXPipeline (this is the correct class name in diffusers >= 0.33.1)
try:
    from diffusers import LTXPipeline
except ImportError:
    from diffusers.pipelines.ltx import LTXPipeline
from diffusers.utils import export_to_video
from pyngrok import ngrok
print("Diffusers version in Cell 3:", __import__("diffusers").__version__)
print("LTXPipeline loaded:", LTXPipeline)

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    kaggle_user = user_secrets.get_secret("KAGGLE_USERNAME")
except:
    kaggle_user = os.environ.get("KAGGLE_USERNAME", "__KAGGLE_USERNAME_PLACEHOLDER__")

backend_url = "https://alonejutt2288--cartoon-backend-fastapi-modal-app.modal.run/session/register-url"

try:
    print("Loading LTX-Video pipeline on GPU...")
    pipe = LTXPipeline.from_pretrained("Lightricks/LTX-Video", torch_dtype=torch.float16)
    pipe = pipe.to("cuda")
    print("LTX-Video ready on GPU!")

    app = FastAPI()
    last_activity = time.time()
    IDLE_TIMEOUT = 8 * 60 * 60

    class GenerateRequest(BaseModel):
        prompt: str
        scene_id: str
        account_id: str

    @app.post("/generate")
    def generate_video(request: GenerateRequest):
        global last_activity
        last_activity = time.time()
        print(f"Generating: {request.prompt}")
        try:
            with torch.backends.cuda.sdp_kernel(enable_flash=False, enable_math=True, enable_mem_efficient=False):
                result = pipe(
                    prompt=request.prompt,
                    negative_prompt="low quality, worst quality, deformed, distorted, disfigured, motion smear, motion artifacts, fused fingers, bad anatomy, weird hand, ugly",
                    width=512, height=288,
                    num_frames=25,
                    num_inference_steps=40,
                    guidance_scale=3.5,
                    generator=torch.Generator("cuda").manual_seed(42),
                )
            frames = result.frames[0]
            output_file = f"/kaggle/working/{request.scene_id}.mp4"
            export_to_video(frames, output_file, fps=8)
            with open(output_file, "rb") as f:
                video_b64 = base64.b64encode(f.read()).decode()
            last_activity = time.time()
            return {"status": "success", "video_path": output_file, "video_b64": video_b64}
        except Exception as e:
            return JSONResponse(status_code=500, content={"status": "error", "message": str(e)})

    @app.get("/health")
    def health():
        return {"status": "alive", "model": "LTX-Video"}

    def idle_checker():
        global last_activity
        while True:
            time.sleep(30)
            if time.time() - last_activity > IDLE_TIMEOUT:
                print("Idle 10 min. Shutting down...")
                os._exit(0)

    threading.Thread(target=idle_checker, daemon=True).start()
    threading.Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000), daemon=True).start()
    print("FastAPI server ready on port 8000!")

    # Start ngrok tunnel & Register with backend
    try:
        user_secrets = UserSecretsClient()
        ngrok_token = user_secrets.get_secret("NGROK_TOKEN")
    except:
        ngrok_token = os.environ.get("NGROK_TOKEN", "__NGROK_TOKEN_PLACEHOLDER__")

    if ngrok_token:
        ngrok.set_auth_token(ngrok_token)

    public_url = ngrok.connect(8000).public_url
    print(f"Ngrok URL: {public_url}")

    resp = requests.post(backend_url, json={"account_username": kaggle_user, "url": public_url})
    print("Registered:", resp.json())

    print("System fully ready! Waiting for generation requests...")
    while True:
        time.sleep(60)

except Exception as e:
    error_msg = traceback.format_exc()
    print("CRASH ERROR:", error_msg)
    requests.post(backend_url, json={"account_username": kaggle_user, "url": f"ERROR: {str(e)} - {error_msg[-200:]}"})
    raise e
